# Iteration 3 — Multimodal TS Forecasting

**Stages:**
- **Stage 1:** Text source ablation — template / LLM / random on ETTh1 (80 runs)
- **Stage 2:** Full sweep — Tier 2 datasets (Weather, ExchangeRate) + F8/F10 validation on ETTh1 (360 runs)

**Runtime:** MacBook Pro M5 Pro, 24 GB unified memory (MPS backend)  
**Registry:** `results/iteration3_registry.json`  
**Embeddings:** `embeddings/{source}/{dataset}_{split}_minilm.npy`

## 0. Setup

In [1]:
import json
import os
import sys
import glob
import re
import subprocess
from pathlib import Path
from datetime import datetime
import pandas as pd

REPO_ROOT = Path('/Users/egorabrosimov/Projects/multimodality/multimodal_TS_research')
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

REGISTRY_PATH = Path('results/iteration3_registry.json')
RESULTS_DIR   = Path('experiments/results')
CONFIGS_DIR   = Path('experiments/configs')
EMB_TEMPLATE  = Path('embeddings/template')
EMB_LLM       = Path('embeddings/llm')

for d in [REGISTRY_PATH.parent, RESULTS_DIR, CONFIGS_DIR, EMB_TEMPLATE, EMB_LLM]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Working dir : {os.getcwd()}')
print(f'Registry    : {REGISTRY_PATH}')

Working dir : /Users/egorabrosimov/Projects/multimodality/multimodal_TS_research
Registry    : results/iteration3_registry.json


## 1. Registry helpers

In [2]:
def load_registry():
    if REGISTRY_PATH.exists():
        with open(REGISTRY_PATH) as f:
            return json.load(f)
    return []


def is_already_done(exp_name: str) -> bool:
    """Return True if this experiment name already has an entry in the registry."""
    return any(r['name'] == exp_name for r in load_registry())


print(f'Registry has {len(load_registry())} entries.')

Registry has 412 entries.


## 2. Run helper

In [3]:
import yaml
from run_experiment import run, load_config, apply_overrides


def run_iter3(config_path: str, skip_if_done: bool = True, overrides: list = None,
              run_idx: int = None, total: int = None):
    """
    Run a single Iter 3 experiment.
    - Skips if experiment name already in results/iteration3_registry.json.
    - run_experiment.run() handles all registry writes automatically.
    - Restart-safe: re-run this cell after any interruption to resume the queue.
    """
    config   = load_config(config_path)
    if overrides:
        config = apply_overrides(config, overrides)

    exp_name = config.get('name', 'experiment')
    counter  = f'[{run_idx}/{total}]  ' if run_idx is not None else ''

    if skip_if_done and is_already_done(exp_name):
        print(f'{counter}SKIP (already done): {exp_name}')
        return None

    print(f'\n{"=" * 70}')
    print(f'{counter}{exp_name}')
    print(f'{"=" * 70}')
    return run(config_path, overrides=overrides)


print('Run helper ready.')

Run helper ready.


## 3. Stage 1 — Text Source Ablation

**Goal:** determine the best text source before committing to the full sweep.

| Model | Sources | Fracs | Horizons | Seeds | Runs |
|---|---|---|---|---|---|
| GatedFusion | template, llm, random | 100%, 10% | 96, 336 | 3 | 36 |
| FiLMFusion | template, llm, random | 100%, 10% | 96, 336 | 3 | 36 |
| BERTForecaster | template, llm | 100%, 10% | 96, 336 | 1 | 8 |

**Dataset:** ETTh1 only. **Total: 80 runs.**

**Online vs offline:**
- `template` + `random` sources run **online** — no pre-encoded embeddings needed.
  The model generates descriptions on-the-fly using `generate_ts_description`.
- `llm` source requires **offline pre-encoded** LLM embeddings (see §3b).
  Template and random runs can start immediately; LLM runs are gated on embedding generation.

### 3a. Generate Stage 1 configs

In [4]:
# Generates d_*.yaml for GatedFusion + FiLMFusion (template/llm/random)
# and BERTForecaster (template/llm only).
# No --emb_dir here — template+random run online; LLM configs will be
# regenerated with --emb_dir after embeddings are ready (§3b).
!python utils/experiment/generate_configs.py --track d_series

d_configs = sorted(glob.glob('experiments/configs/d_*.yaml'))
print(f'\nStage 1 configs: {len(d_configs)} total')

D-series: wrote 80 configs to experiments/configs/

Stage 1 configs: 80 total


### 3b. Run template + random sources (online mode)

These can start immediately. LLM configs (d_*_srclllm_*.yaml) are skipped
until offline embeddings are available.

In [5]:
# Filter to template + random only (skip llm until embeddings are ready)
tr_configs = [c for c in d_configs if '_srcllm_' not in os.path.basename(c)]
total = len(tr_configs)

already = sum(1 for c in tr_configs
              if is_already_done(load_config(c).get('name', '')))
print(f'Template+random queue: {total} | done: {already} | remaining: {total - already}\n')

for i, cfg in enumerate(tr_configs, 1):
    run_iter3(cfg, run_idx=i, total=total)

Template+random queue: 52 | done: 52 | remaining: 0

[1/52]  SKIP (already done): d_bertforecaster_etth1_srctemplate_f100_h336_s2024
[2/52]  SKIP (already done): d_bertforecaster_etth1_srctemplate_f100_h96_s2024
[3/52]  SKIP (already done): d_bertforecaster_etth1_srctemplate_f10_h336_s2024
[4/52]  SKIP (already done): d_bertforecaster_etth1_srctemplate_f10_h96_s2024
[5/52]  SKIP (already done): d_filmfusion_etth1_srcrandom_f100_h336_s2024
[6/52]  SKIP (already done): d_filmfusion_etth1_srcrandom_f100_h336_s2025
[7/52]  SKIP (already done): d_filmfusion_etth1_srcrandom_f100_h336_s2026
[8/52]  SKIP (already done): d_filmfusion_etth1_srcrandom_f100_h96_s2024
[9/52]  SKIP (already done): d_filmfusion_etth1_srcrandom_f100_h96_s2025
[10/52]  SKIP (already done): d_filmfusion_etth1_srcrandom_f100_h96_s2026
[11/52]  SKIP (already done): d_filmfusion_etth1_srcrandom_f10_h336_s2024
[12/52]  SKIP (already done): d_filmfusion_etth1_srcrandom_f10_h336_s2025
[13/52]  SKIP (already done): d_filmfusio

### 3c. LLM text source — Phi-4-mini embedding generation

**Model:** `mlx-community/Phi-4-mini-instruct-4bit` (Microsoft, ~2.5 GB, ~130 tok/s on M5 Pro)

**Pipeline:**
1. Load Phi-4-mini via mlx-lm
2. For each ETTh1 window, feed the template statistics as context → LLM generates a 1–2 sentence interpretation
3. Encode LLM outputs with MiniLM-L6 → save `embeddings/llm/ETTh1_{split}_minilm.npy`
4. Run cosine check vs template embeddings (must be < 0.95)

**One-time cost:** ~14k windows × ~0.4s ≈ 1.5h for all splits.

In [6]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from mlx_lm import load as mlx_load, generate as mlx_generate
from layers.TextEncoder import TextEncoder, generate_ts_description
from utils.data_provider.data_loader import Dataset_ETT_hour

PHI4_MODEL_ID    = 'mlx-community/Phi-4-mini-instruct-4bit'
ETTH1_SPLITS     = ['train', 'val', 'test']
CHECKPOINT_EVERY = 50   # save partial .npy every N batches

# ── Load Phi-4-mini (downloads ~2.5 GB on first run) ──────────────────────────
print(f'Loading {PHI4_MODEL_ID} ...')
phi_model, phi_tokenizer = mlx_load(PHI4_MODEL_ID)
print('Phi-4-mini ready.')

# ── Load MiniLM-L6 encoder on MPS ─────────────────────────────────────────────
minilm = TextEncoder()
minilm._model = minilm._model.to('mps')


def phi4_describe(template_text: str, dataset_name: str) -> str:
    """Feed template stats into Phi-4-mini → richer 1–2 sentence interpretation."""
    messages = [
        {
            'role': 'system',
            'content': (
                'You are a concise time series analyst. '
                'Given window statistics, write 1-2 sentences interpreting '
                'the pattern and what might drive it. Do not restate the numbers.'
            ),
        },
        {'role': 'user', 'content': f'Dataset: {dataset_name}. {template_text}'},
    ]
    prompt = phi_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(phi_model, phi_tokenizer, prompt=prompt,
                        max_tokens=80, verbose=False).strip()


# Dataset_ETT_hour takes size=(seq_len, label_len, pred_len) not individual kwargs
DATASET_KWARGS = dict(
    root_path='./dataset/ETT-small/',
    data_path='ETTh1.csv',
    features='M',
    target='OT',
    freq='h',
    size=(96, 24, 96),   # (seq_len, label_len, pred_len)
)

# ── Pre-compute train-split stats for consistent regime labels across splits ───
print('Computing train-split stats ...')
train_ds = Dataset_ETT_hour(**DATASET_KWARGS, flag='train')
stds, slopes = [], []
for i in range(len(train_ds)):
    x, *_ = train_ds[i]
    s = torch.tensor(x).mean(-1).numpy()
    stds.append(float(np.std(s)))
    slopes.append(float((s[-1] - s[0]) / max(len(s) - 1, 1)))
train_stats = {
    'std_p80':   float(np.percentile(stds, 80)),
    'slope_p80': float(np.percentile(np.abs(slopes), 80)),
}
print(f'  std_p80={train_stats["std_p80"]:.4f}  slope_p80={train_stats["slope_p80"]:.6f}')


# ── Encode each split (resume-safe) ───────────────────────────────────────────
for split in ETTH1_SPLITS:
    out_path     = Path(f'embeddings/llm/ETTh1_{split}_minilm.npy')
    partial_path = out_path.with_suffix('.partial.npy')

    if out_path.exists():
        print(f'Already encoded: {out_path}')
        continue

    if partial_path.exists():
        checkpoint = np.load(partial_path)
        done_count = checkpoint.shape[0]
        all_embs   = [checkpoint]
        print(f'Resuming ETTh1/{split} from {done_count} windows ...')
    else:
        done_count = 0
        all_embs   = []

    ds     = Dataset_ETT_hour(**DATASET_KWARGS, flag=split)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)

    window_count = 0
    new_batches  = 0

    for batch in tqdm(loader, desc=f'ETTh1/{split}'):
        x_enc, x_dec, x_mark_enc, x_mark_dec = batch[:4]
        batch_size = x_enc.shape[0]

        # Skip batches fully covered by checkpoint
        if window_count + batch_size <= done_count:
            window_count += batch_size
            continue
        window_count += batch_size

        texts_template = generate_ts_description(
            x_enc, dataset_name='ETTh1', pred_len=96,
            x_mark_enc=x_mark_enc, dataset_stats=train_stats,
        )
        texts_llm = [phi4_describe(t, 'ETTh1') for t in texts_template]

        with torch.no_grad():
            emb = minilm(texts_llm).cpu().numpy()   # [B, 384]
        all_embs.append(emb)
        new_batches += 1

        if new_batches % CHECKPOINT_EVERY == 0:
            np.save(partial_path, np.concatenate(all_embs, axis=0))
            print(f'  Checkpoint: {sum(e.shape[0] for e in all_embs)} windows')

    embs = np.concatenate(all_embs, axis=0)
    np.save(out_path, embs)
    partial_path.unlink(missing_ok=True)
    print(f'Saved: {out_path}  shape={embs.shape}')

/Users/egorabrosimov/.local/share/mamba/envs/multimodality_experiment_3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading mlx-community/Phi-4-mini-instruct-4bit ...


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 59074.70it/s]
This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


Phi-4-mini ready.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9254.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing train-split stats ...
  std_p80=0.4020  slope_p80=0.005786
Already encoded: embeddings/llm/ETTh1_train_minilm.npy
Already encoded: embeddings/llm/ETTh1_val_minilm.npy
Already encoded: embeddings/llm/ETTh1_test_minilm.npy


In [7]:
# ── Cosine similarity check: LLM vs template embeddings ───────────────────────
# Run after LLM embeddings are generated. Mean > 0.95 → LLM descriptions
# are too similar to template → LLM source carries no additional signal.
import numpy as np
import torch

llm_path  = 'embeddings/llm/ETTh1_train_minilm.npy'
tmpl_path = 'embeddings/template/ETTh1_train_minilm.npy'

if Path(llm_path).exists() and Path(tmpl_path).exists():
    llm  = torch.tensor(np.load(llm_path)[:500]).float()
    tmpl = torch.tensor(np.load(tmpl_path)[:500]).float()
    llm_n  = torch.nn.functional.normalize(llm,  dim=-1)
    tmpl_n = torch.nn.functional.normalize(tmpl, dim=-1)
    cross_sim = (llm_n * tmpl_n).sum(-1)          # pairwise cosine [N]
    print(f'LLM ↔ template cosine similarity:  mean={cross_sim.mean():.4f}  std={cross_sim.std():.4f}')
    print('→ Proceed if mean < 0.95')
else:
    print('Embeddings not yet generated — run §3c first.')

LLM ↔ template cosine similarity:  mean=0.5392  std=0.0612
→ Proceed if mean < 0.95


In [8]:
# ── Regenerate d_series configs with LLM emb_dir ──────────────────────────────
# Run after LLM embeddings pass the cosine check.
!python utils/experiment/generate_configs.py --track d_series --emb_dir embeddings/llm

llm_configs = sorted(glob.glob('experiments/configs/d_*_srcllm_*.yaml'))
print(f'LLM configs: {len(llm_configs)}')

D-series: wrote 80 configs to experiments/configs/
LLM configs: 28


In [9]:
# ── Run LLM source experiments ─────────────────────────────────────────────────
total = len(llm_configs)
already = sum(1 for c in llm_configs
              if is_already_done(load_config(c).get('name', '')))
print(f'LLM queue: {total} | done: {already} | remaining: {total - already}\n')

for i, cfg in enumerate(llm_configs, 1):
    run_iter3(cfg, run_idx=i, total=total)

LLM queue: 28 | done: 28 | remaining: 0

[1/28]  SKIP (already done): d_bertforecaster_etth1_srcllm_f100_h336_s2024
[2/28]  SKIP (already done): d_bertforecaster_etth1_srcllm_f100_h96_s2024
[3/28]  SKIP (already done): d_bertforecaster_etth1_srcllm_f10_h336_s2024
[4/28]  SKIP (already done): d_bertforecaster_etth1_srcllm_f10_h96_s2024
[5/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f100_h336_s2024
[6/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f100_h336_s2025
[7/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f100_h336_s2026
[8/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f100_h96_s2024
[9/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f100_h96_s2025
[10/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f100_h96_s2026
[11/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f10_h336_s2024
[12/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f10_h336_s2025
[13/28]  SKIP (already done): d_filmfusion_etth1_srcllm_f10_h336_s2026
[14/28]  SKIP (already don

## 4. Stage 1 Analysis

Compare template / llm / random for GatedFusion and FiLMFusion.  
The winning text source is locked in before Stage 2 launches.

**Interpretation guide:**
- `template < random` on MSE → text semantics contribute (not just regularization)
- `llm ≤ template` → richer descriptions add no value over template stats
- `random ≈ template` → text acts as regularizer only

In [10]:
registry = load_registry()

d_rows = []
for r in registry:
    if not r['name'].startswith('d_'):
        continue
    d_rows.append({
        'model':    r['model'],
        'fraction': r.get('train_fraction', 1.0),
        'pred_len': r['pred_len'],
        'seed':     r.get('seed'),
        'text_src': r.get('text_source', '?'),
        'MAE':      r['metrics'].get('mae'),
        'MSE':      r['metrics'].get('mse'),
    })

df_d = pd.DataFrame(d_rows)
print(f'Stage 1 runs loaded: {len(df_d)}')

# Mean ± std across seeds, grouped by model / source / fraction / horizon
summary = (
    df_d.groupby(['model', 'text_src', 'fraction', 'pred_len'])[['MAE', 'MSE']]
    .agg(['mean', 'std'])
    .round(4)
)
summary

Stage 1 runs loaded: 80


MAE             MSE        
                                             mean     std    mean     std
model          text_src fraction pred_len                                
BERTForecaster llm      0.1      96        0.8045     NaN  1.1198     NaN
                                 336       0.8078     NaN  1.1186     NaN
                        1.0      96        0.8001     NaN  1.0879     NaN
                                 336       0.8027     NaN  1.0817     NaN
               template 0.1      96        0.8028     NaN  1.1182     NaN
                                 336       0.8067     NaN  1.1172     NaN
                        1.0      96        0.8108     NaN  1.0824     NaN
                                 336       0.8251     NaN  1.0919     NaN
FiLMFusion     llm      0.1      96        0.4221  0.0045  0.3989  0.0032
                                 336       0.4536  0.0017  0.4425  0.0021
                        1.0      96        0.4076  0.0010  0.3814  0.0042
                                 336       0.4521  0.0308  0.4431  0.0446
               random   0.1      96        0.4899  0.0023  0.5400  0.0038
                                 336       0.4974  0.0026  0.5320  0.0024
                        1.0      96        0.4305  0.0028  0.4063  0.0044
                                 336       0.4866  0.0205  0.4950  0.0413
               template 0.1      96        0.4181  0.0037  0.3923  0.0045
                                 336       0.4497  0.0007  0.4365  0.0003
                        1.0      96        0.4028  0.0012  0.3743  0.0033
                                 336       0.4588  0.0263  0.4564  0.0403
GatedFusion    llm      0.1      96        0.4202  0.0018  0.3949  0.0027
                                 336       0.4676  0.0014  0.4598  0.0024
                        1.0      96        0.4055  0.0005  0.3802  0.0050
                                 336       0.4499  0.0254  0.4403  0.0340
               random   0.1      96        0.4236  0.0016  0.3987  0.0012
                                 336       0.4700  0.0017  0.4637  0.0028
                        1.0      96        0.4095  0.0011  0.3849  0.0063
                                 336       0.4528  0.0265  0.4481  0.0396
               template 0.1      96        0.4203  0.0020  0.3949  0.0022
                                 336       0.4679  0.0018  0.4603  0.0027
                        1.0      96        0.4061  0.0008  0.3816  0.0050
                                 336       0.4518  0.0293  0.4423  0.0372

In [11]:
# Aggregate across fractions + horizons — best source per model
fusion_models = ['GatedFusion', 'FiLMFusion']
agg = (
    df_d[df_d['model'].isin(fusion_models)]
    .groupby(['model', 'text_src'])[['MAE', 'MSE']]
    .mean()
    .round(4)
    .reset_index()
    .sort_values(['model', 'MSE'])
)
print(agg.to_string(index=False))

# ← Set after reviewing the table above
TEXT_SOURCE_WINNER = 'template'
print(f'\nText source winner locked: {TEXT_SOURCE_WINNER}')

      model text_src    MAE    MSE
 FiLMFusion template 0.4324 0.4149
 FiLMFusion      llm 0.4339 0.4165
 FiLMFusion   random 0.4761 0.4933
GatedFusion      llm 0.4358 0.4188
GatedFusion template 0.4365 0.4198
GatedFusion   random 0.4390 0.4239

Text source winner locked: template


## 5. Stage 2 — Full Sweep

| Sub-task | Models | Datasets | Runs |
|---|---|---|---|
| Tier 2 sweep | DLinear, PatchTST, GatedFusion, FiLMFusion, EnsembleFusion | Weather, ExchangeRate | 300 |
| F8/F10 validation | CrossAttentionFusion, ResidualCorrection | ETTh1 | 60 |

**Config:** winner text source from Stage 1, horizons {96, 336}, fractions {100%, 50%, 25%, 10%, 5%}, 3 seeds.

**Gate:** complete Stage 1 analysis and set `TEXT_SOURCE_WINNER` before running this section.

### 5a. Offline embedding encoding — Tier 2 datasets

Encode template embeddings for Weather and ExchangeRate (one-time, ~5 min each on MPS).

In [12]:
TIER2_DATASETS = [
    dict(name='Weather',      root='./dataset/weather/',       csv='weather.csv',       freq='h'),
    dict(name='ExchangeRate', root='./dataset/exchange_rate/', csv='exchange_rate.csv', freq='d'),
]

for ds in TIER2_DATASETS:
    std_p80 = slope_p80 = None
    for split in ['train', 'val', 'test']:
        out = f'embeddings/template/{ds["name"]}_{split}_minilm.npy'
        if Path(out).exists():
            print(f'Already encoded: {out}')
            continue

        cmd = [
            'python', 'utils/text/encode_descriptions.py',
            '--dataset', 'custom',
            '--split', split,
            '--root_path', ds['root'],
            '--data_path', ds['csv'],
            '--freq', ds['freq'],
            '--out', out,
        ]
        if split != 'train' and std_p80 is not None:
            cmd += ['--std_p80', str(std_p80), '--slope_p80', str(slope_p80)]

        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError(f'Encoding failed: {ds["name"]}/{split}')

        # Parse train stats for reuse in val/test
        if split == 'train':
            m = re.search(r'std_p80=(\S+).*slope_p80=(\S+)', result.stdout)
            if m:
                std_p80, slope_p80 = float(m.group(1)), float(m.group(2))
                print(f'  Train stats: std_p80={std_p80:.4f}  slope_p80={slope_p80:.4f}')

        print(f'Saved: {out}')

Already encoded: embeddings/template/Weather_train_minilm.npy
Already encoded: embeddings/template/Weather_val_minilm.npy
Already encoded: embeddings/template/Weather_test_minilm.npy
Already encoded: embeddings/template/ExchangeRate_train_minilm.npy
Already encoded: embeddings/template/ExchangeRate_val_minilm.npy
Already encoded: embeddings/template/ExchangeRate_test_minilm.npy


### 5b. Generate Stage 2 configs

In [13]:
# ── Tier 2: Weather + ExchangeRate, all CORE_MODELS ───────────────────────────
!python utils/experiment/generate_configs.py \
    --track tier2 \
    --text_source {TEXT_SOURCE_WINNER} \
    --emb_dir embeddings/template

# ── F8/F10 validation: ETTh1 only ─────────────────────────────────────────────
!python utils/experiment/generate_configs.py \
    --track tier1 \
    --models CrossAttentionFusion ResidualCorrection \
    --datasets ETTh1 \
    --text_source {TEXT_SOURCE_WINNER} \
    --emb_dir embeddings/template

t2_configs  = sorted(glob.glob('experiments/configs/t2_*.yaml'))
f8f10_configs = sorted(glob.glob('experiments/configs/t1_cross*.yaml'))
f8f10_configs += sorted(glob.glob('experiments/configs/t1_residual*.yaml'))

print(f'Tier 2 configs     : {len(t2_configs)}')
print(f'F8/F10 configs     : {len(f8f10_configs)}')
print(f'Stage 2 total      : {len(t2_configs) + len(f8f10_configs)}')

Tier 2: wrote 300 configs to experiments/configs/
Tier 1: wrote 60 configs to experiments/configs/
Tier 2 configs     : 300
F8/F10 configs     : 60
Stage 2 total      : 360


### 5c. Run Stage 2

Restart-safe — already-done experiments are skipped automatically.

In [14]:
# Priority: F8/F10 first (validate fixes on familiar ETTh1),
# then Tier 2 in dataset order.
run_queue = f8f10_configs + t2_configs
TOTAL = len(run_queue)

already = sum(1 for c in run_queue
              if is_already_done(load_config(c).get('name', '')))
print(f'Stage 2 queue: {TOTAL} | done: {already} | remaining: {TOTAL - already}\n')

Stage 2 queue: 360 | done: 332 | remaining: 28



In [15]:
# Re-run this cell after any interruption — done experiments are skipped.
for i, cfg in enumerate(run_queue, 1):
    run_iter3(cfg, run_idx=i, total=TOTAL)

[1/360]  SKIP (already done): t1_crossattentionfusion_etth1_f100_h336_s2024
[2/360]  SKIP (already done): t1_crossattentionfusion_etth1_f100_h336_s2025
[3/360]  SKIP (already done): t1_crossattentionfusion_etth1_f100_h336_s2026
[4/360]  SKIP (already done): t1_crossattentionfusion_etth1_f100_h96_s2024
[5/360]  SKIP (already done): t1_crossattentionfusion_etth1_f100_h96_s2025
[6/360]  SKIP (already done): t1_crossattentionfusion_etth1_f100_h96_s2026
[7/360]  SKIP (already done): t1_crossattentionfusion_etth1_f10_h336_s2024
[8/360]  SKIP (already done): t1_crossattentionfusion_etth1_f10_h336_s2025
[9/360]  SKIP (already done): t1_crossattentionfusion_etth1_f10_h336_s2026
[10/360]  SKIP (already done): t1_crossattentionfusion_etth1_f10_h96_s2024
[11/360]  SKIP (already done): t1_crossattentionfusion_etth1_f10_h96_s2025
[12/360]  SKIP (already done): t1_crossattentionfusion_etth1_f10_h96_s2026
[13/360]  SKIP (already done): t1_crossattentionfusion_etth1_f25_h336_s2024
[14/360]  SKIP (alrea

Training:  40%|████      | 6/15 [48:53<1:13:19, 488.88s/epoch, train=0.5238, val=0.5740]


Early stopping at epoch 7


Testing: 100%|██████████| 319/319 [00:25<00:00, 12.46batch/s]


Test | MAE=0.2837  MSE=0.2485  RMSE=0.4985

Done. Results saved to experiments/results/t2_patchtst_weather_f100_h336_s2026_20260414_220916

[334/360]  t2_patchtst_weather_f100_h96_s2024

Experiment : t2_patchtst_weather_f100_h96_s2024
Model      : PatchTST
Dataset    : custom
pred_len   : 96
Results in : experiments/results/t2_patchtst_weather_f100_h96_s2024_20260414_225838

Using GPU: Apple MPS
Model: PatchTST | Total params: 1,581,024 | Trainable: 1,581,024


...

Test | MAE=0.2262  MSE=0.1739  RMSE=0.4170

Done. Results saved to experiments/results/t2_patchtst_weather_f5_h96_s2025_20260415_080339

[360/360]  t2_patchtst_weather_f5_h96_s2026

Experiment : t2_patchtst_weather_f5_h96_s2026
Model      : PatchTST
Dataset    : custom
pred_len   : 96
Results in : experiments/results/t2_patchtst_weather_f5_h96_s2026_20260415_080955

Using GPU: Apple MPS
Model: PatchTST | Total params: 1,581,024 | Trainable: 1,581,024


Training:  87%|████████▋ | 13/15 [06:27<00:59, 29.79s/epoch, train=0.2651, val=0.4097]


Early stopping at epoch 14


Testing: 100%|██████████| 327/327 [00:13<00:00, 24.01batch/s]


Test | MAE=0.2253  MSE=0.1726  RMSE=0.4154

Done. Results saved to experiments/results/t2_patchtst_weather_f5_h96_s2026_20260415_080955


## 6. Results Summary

In [16]:
registry = load_registry()

rows = []
for r in registry:
    rows.append({
        'name':       r['name'],
        'model':      r['model'],
        'dataset':    r['dataset'],
        'fraction':   r.get('train_fraction', 1.0),
        'pred_len':   r['pred_len'],
        'seed':       r.get('seed'),
        'text_src':   r.get('text_source', '-'),
        'MAE':        r['metrics'].get('mae'),
        'MSE':        r['metrics'].get('mse'),
    })

df = pd.DataFrame(rows)
print(f'Total runs in registry: {len(df)}')
df.sort_values(['dataset', 'model', 'fraction', 'pred_len']).head(30)

Total runs in registry: 468


,name,model,dataset,fraction,pred_len,seed,text_src,MAE,MSE
3,d_bertforecaster_etth1_srctemplate_f10_h96_s2024,BERTForecaster,ETTh1,0.10,96,2024,template,0.802841,1.118243
55,d_bertforecaster_etth1_srcllm_f10_h96_s2024,BERTForecaster,ETTh1,0.10,96,2024,llm,0.804532,1.119842
2,d_bertforecaster_etth1_srctemplate_f10_h336_s2024,BERTForecaster,ETTh1,0.10,336,2024,template,0.806694,1.117200
54,d_bertforecaster_etth1_srcllm_f10_h336_s2024,BERTForecaster,ETTh1,0.10,336,2024,llm,0.807795,1.118639
1,d_bertforecaster_etth1_srctemplate_f100_h96_s2024,BERTForecaster,ETTh1,1.00,96,2024,template,0.810761,1.082436
53,d_bertforecaster_etth1_srcllm_f100_h96_s2024,BERTForecaster,ETTh1,1.00,96,2024,llm,0.800076,1.087918
0,d_bertforecaster_etth1_srctemplate_f100_h336_s...,BERTForecaster,ETTh1,1.00,336,2024,template,0.825150,1.091885
52,d_bertforecaster_etth1_srcllm_f100_h336_s2024,BERTForecaster,ETTh1,1.00,336,2024,llm,0.802713,1.081694
107,t1_crossattentionfusion_etth1_f5_h96_s2024,CrossAttentionFusion,ETTh1,0.05,96,2024,template,0.457321,0.441608
108,t1_crossattentionfusion_etth1_f5_h96_s2025,CrossAttentionFusion,ETTh1,0.05,96,2025,template,0.455001,0.438114


In [17]:
# Mean ± std across seeds — main results table
iter3_df = df[~df['name'].str.startswith('d_')]  # exclude Stage 1 ablation

summary = (
    iter3_df
    .groupby(['model', 'dataset', 'fraction', 'pred_len'])[['MAE', 'MSE']]
    .agg(['mean', 'std'])
    .round(4)
)
summary

MAE             MSE        
                                                  mean     std    mean     std
model                dataset fraction pred_len                                
CrossAttentionFusion ETTh1   0.05     96        0.4577  0.0029  0.4422  0.0044
                                      336       0.5400  0.0129  0.5831  0.0241
                             0.10     96        0.4441  0.0046  0.4247  0.0063
                                      336       0.4907  0.0011  0.4999  0.0025
                             0.25     96        0.4316  0.0037  0.4092  0.0049
...                                                ...     ...     ...     ...
ResidualCorrection   ETTh1   0.25     336       0.4798  0.0006  0.4751  0.0012
                             0.50     96        0.4225  0.0020  0.3996  0.0015
                                      336       0.4722  0.0060  0.4681  0.0072
                             1.00     96        0.4178  0.0047  0.3947  0.0062
                                      336       0.4471  0.0058  0.4397  0.0050

[70 rows x 4 columns]

In [18]:
# Degradation analysis: MSE at 5% relative to 100% per model/dataset/horizon
# Lower ratio → more graceful degradation
agg_mean = (
    iter3_df
    .groupby(['model', 'dataset', 'pred_len', 'fraction'])['MSE']
    .mean()
    .reset_index()
)

full  = agg_mean[agg_mean['fraction'] == 1.0].rename(columns={'MSE': 'MSE_full'})
low   = agg_mean[agg_mean['fraction'] == 0.05].rename(columns={'MSE': 'MSE_low'})

degrad = full.merge(low, on=['model', 'dataset', 'pred_len'])
degrad['ratio_5pct'] = (degrad['MSE_low'] / degrad['MSE_full']).round(3)
degrad = degrad.sort_values(['dataset', 'pred_len', 'ratio_5pct'])

print('MSE degradation ratio at 5% vs 100% data (lower = more graceful):')
degrad[['model', 'dataset', 'pred_len', 'MSE_full', 'MSE_low', 'ratio_5pct']]

MSE degradation ratio at 5% vs 100% data (lower = more graceful):


,model,dataset,pred_len,MSE_full,MSE_low,ratio_5pct
0,CrossAttentionFusion,ETTh1,96,0.401795,0.442212,1.101
12,ResidualCorrection,ETTh1,96,0.394708,0.434771,1.101
1,CrossAttentionFusion,ETTh1,336,0.461573,0.583118,1.263
13,ResidualCorrection,ETTh1,336,0.439676,0.566294,1.288
10,PatchTST,custom,96,0.136869,0.195407,1.428
4,EnsembleFusion,custom,96,0.140239,0.228433,1.629
8,GatedFusion,custom,96,0.130027,0.222912,1.714
6,FiLMFusion,custom,96,0.130620,0.251348,1.924
2,DLinear,custom,96,0.129171,0.790988,6.124
11,PatchTST,custom,336,0.322491,0.378869,1.175
